# 1. Extracción de estimados

Este cuaderno contiene todo el proceso visible: inventario recursivo, selección de archivos, regla de semana faltante, lectura de la cache de Excel, construcción de H1-H5, controles y EDA.

Regla importante: si no existe el archivo de la semana solicitada, se usa el siguiente archivo disponible del mismo año. Por ejemplo, 2026-S02 usa el archivo 2026-S03 y queda marcado en `archivo_es_posterior`.

In [ ]:
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from zipfile import BadZipFile, ZipFile
import re
import unicodedata

import numpy as np
import pandas as pd
from lxml import etree

ROOT_PROYECTO = next(r for r in [Path.cwd(), *Path.cwd().parents] if (r / 'Datos' / 'Estimados semanales').is_dir())
RAIZ_ESTIMADOS = ROOT_PROYECTO / 'Datos' / 'Estimados semanales'
CARPETA_SALIDA = RAIZ_ESTIMADOS / 'datos extraidos'
NUMERO_HORIZONTES = 5
INCLUIR_AJUSTADOS = False
NS_EXCEL = 'http://schemas.openxmlformats.org/spreadsheetml/2006/main'
EXTENSIONES_EXCEL = {'.xlsx', '.xlsm', '.xlsb', '.xls'}
print('Raíz:', ROOT_PROYECTO)
print('Entrada:', RAIZ_ESTIMADOS)
print('Salida:', CARPETA_SALIDA)

## 1. Inventario y plan de solicitudes

Primero se recorre toda la carpeta, no solo 2026. Se conserva un archivo principal por año-semana y los duplicados o ajustados quedan en tablas de control. El plan separa `semana_solicitada` de `semana_archivo`: esa separación es la que evita perder 2026-S02.

In [ ]:
def normalizar_texto(valor):
    if pd.isna(valor): return ''
    texto = str(valor).replace('\xa0', ' ').strip()
    texto = ''.join(c for c in unicodedata.normalize('NFKD', texto) if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', texto).upper()

def convertir_entero(valor):
    try:
        if pd.isna(valor): return pd.NA
        numero = float(str(valor).replace(',', '.'))
        return int(numero) if numero.is_integer() else pd.NA
    except (TypeError, ValueError):
        return pd.NA

def ano_de_carpeta(ruta):
    return next((int(p) for p in ruta.parts if re.fullmatch(r'20\d{2}', p)), pd.NA)

def analizar_archivo(ruta):
    nombre = ruta.stem
    normalizado = normalizar_texto(nombre)
    match = re.search(r'\bSEM(?:ANA)?\s*[-_ ]*(\d{1,2})\b', normalizado)
    semana = int(match.group(1)) if match else pd.NA
    anos = [int(x) for x in re.findall(r'(?<!\d)(20\d{2})(?!\d)', nombre)]
    ano = anos[0] if anos else ano_de_carpeta(ruta)
    return {'archivo': ruta.name, 'ruta_absoluta': str(ruta), 'ruta_relativa': str(ruta.relative_to(RAIZ_ESTIMADOS)), 'carpeta': ruta.parent.name, 'ano_inicio': ano, 'semana_inicio': semana, 'es_ajustado': bool(re.search(r'\bAJUST|CORRECC|CORREG', normalizado)), 'tiene_marca_envio': 'ENVIO' in normalizado, 'tiene_sello_horario': bool(re.search(r'\d{8}[_ -]\d{6}', nombre)), 'tamano_bytes': ruta.stat().st_size, 'fecha_modificacion': pd.Timestamp(ruta.stat().st_mtime, unit='s')}

def construir_inventario():
    filas = []
    for ruta in sorted(RAIZ_ESTIMADOS.rglob('*')):
        if not ruta.is_file() or ruta.suffix.lower() not in EXTENSIONES_EXCEL or ruta.name.startswith('~$'): continue
        if not re.search(r'ESTIM', normalizar_texto(ruta.name)): continue
        filas.append(analizar_archivo(ruta))
    inventario = pd.DataFrame(filas)
    if inventario.empty: return inventario
    inventario['semana_parseable'] = inventario['semana_inicio'].notna(); inventario['ano_parseable'] = inventario['ano_inicio'].notna()
    inventario['candidato_extraccion'] = inventario['semana_parseable'] & inventario['ano_parseable'] & inventario['semana_inicio'].between(1, 53)
    return inventario.sort_values(['ano_inicio', 'semana_inicio', 'ruta_relativa'], na_position='last').reset_index(drop=True)

def seleccionar_archivos(inventario):
    candidatos = inventario[inventario['candidato_extraccion']].copy(); excluidos = inventario[~inventario['candidato_extraccion']].copy()
    if not INCLUIR_AJUSTADOS:
        excluidos = pd.concat([excluidos, candidatos[candidatos['es_ajustado']]], ignore_index=True); candidatos = candidatos[~candidatos['es_ajustado']].copy()
    candidatos['_prioridad'] = candidatos['tiene_marca_envio'].astype(int) * -2 + candidatos['tiene_sello_horario'].astype(int) * 2
    candidatos = candidatos.sort_values(['ano_inicio', 'semana_inicio', '_prioridad', 'archivo'])
    seleccionados = candidatos.drop_duplicates(['ano_inicio', 'semana_inicio'], keep='first').drop(columns='_prioridad')
    duplicados = candidatos.loc[~candidatos['ruta_absoluta'].isin(seleccionados['ruta_absoluta'])].drop(columns='_prioridad')
    return seleccionados.reset_index(drop=True), duplicados.reset_index(drop=True), excluidos.reset_index(drop=True)

def construir_solicitudes_semanales(seleccionados):
    solicitudes = []
    for ano, grupo in seleccionados.groupby('ano_inicio'):
        grupo = grupo.sort_values('semana_inicio')
        for semana_solicitada in range(1, int(grupo['semana_inicio'].max()) + 1):
            candidatos = grupo[grupo['semana_inicio'].astype(int) >= semana_solicitada]
            if candidatos.empty: continue
            fila = candidatos.iloc[0].copy(); fila['semana_solicitada'] = semana_solicitada
            fila['archivo_es_posterior'] = int(fila['semana_inicio']) > semana_solicitada; fila['diferencia_semanas_archivo'] = int(fila['semana_inicio']) - semana_solicitada; solicitudes.append(fila)
    return pd.DataFrame(solicitudes).reset_index(drop=True)

inventario_archivos = construir_inventario()
archivos_seleccionados, archivos_duplicados, archivos_excluidos = seleccionar_archivos(inventario_archivos)
plan_extraccion = construir_solicitudes_semanales(archivos_seleccionados)
display(inventario_archivos.groupby(['ano_inicio', 'es_ajustado'], dropna=False).size().rename('archivos').reset_index())
display(plan_extraccion.loc[(plan_extraccion['ano_inicio'] == 2026) & (plan_extraccion['semana_solicitada'] <= 3), ['ano_inicio', 'semana_solicitada', 'semana_inicio', 'archivo', 'archivo_es_posterior', 'diferencia_semanas_archivo']])

## 2. Lectura de la cache de Excel

Los archivos Excel se leen directamente desde `pivotCacheRecords`. Si un archivo no expone la cache, se intenta una lectura de hojas como fallback y el método queda registrado.

In [ ]:
def etiqueta(elemento): return etree.QName(elemento).localname

def valor_cache(elemento, compartidos):
    tipo = etiqueta(elemento); valor = elemento.get('v')
    if tipo == 'm': return None
    if tipo == 'x':
        try: return compartidos[int(valor)]
        except (TypeError, ValueError, IndexError): return None
    if tipo == 'n':
        try:
            numero = float(valor); return int(numero) if numero.is_integer() else numero
        except (TypeError, ValueError): return valor
    if tipo == 'b': return valor == '1'
    return valor if valor is not None else ''.join(elemento.itertext()).strip() or None

def resolver_campos(columnas):
    aliases = {normalizar_texto(c): c for c in columnas}
    opciones = {'tipo': ['TIPO', 'TYPE'], 'ano': ['AÑO', 'ANO', 'YEAR'], 'semana': ['SEMANA', 'WEEK'], 'tallos': ['ESTI. EXPORTABLE', 'ESTI EXPORTABLE', 'TALLOS', 'ESTIMADO']}
    encontrados = {k: next((aliases[normalizar_texto(o)] for o in v if normalizar_texto(o) in aliases), None) for k, v in opciones.items()}
    faltantes = [k for k, v in encontrados.items() if v is None]
    if faltantes: raise ValueError(f'Faltan campos: {faltantes}; disponibles: {columnas}')
    return encontrados

def extraer_desde_cache(ruta, anos_interes):
    with ZipFile(ruta) as libro:
        nombres = libro.namelist(); definiciones = sorted(n for n in nombres if 'pivotcache/pivotcachedefinition' in n.lower() and '/_rels/' not in n.lower()); registros = sorted(n for n in nombres if 'pivotcache/pivotcacherecords' in n.lower())
        if not definiciones or not registros: raise ValueError('No hay cache de tabla dinámica')
        definicion = etree.fromstring(libro.read(definiciones[0])); campos_xml = definicion.find(f'{{{NS_EXCEL}}}cacheFields')
        columnas = [c.get('name') or f'campo_{i}' for i, c in enumerate(campos_xml)]; campos = resolver_campos(columnas); compartidos = []
        for campo in campos_xml:
            bloque = campo.find(f'{{{NS_EXCEL}}}sharedItems'); compartidos.append([] if bloque is None else [valor_cache(item, []) for item in bloque])
        indice_tipo = columnas.index(campos['tipo']); indice_ano = columnas.index(campos['ano']); filas = []
        with libro.open(registros[0]) as flujo:
            for _, registro in etree.iterparse(flujo, events=('end',), tag=f'{{{NS_EXCEL}}}r'):
                celdas = list(registro); valores = [valor_cache(celdas[i], compartidos[i]) if i < len(celdas) else None for i in range(len(columnas))]; ano = convertir_entero(valores[indice_ano])
                if normalizar_texto(valores[indice_tipo]) == 'ESTIMADO' and pd.notna(ano) and int(ano) in anos_interes: filas.append(dict(zip(columnas, valores)))
                registro.clear(); padre = registro.getparent()
                while registro.getprevious() is not None: del padre[0]
    tabla = pd.DataFrame(filas, columns=columnas); return tabla.rename(columns={v: k for k, v in campos.items()})

def leer_estimados_archivo(ruta, anos_interes):
    try:
        tabla = extraer_desde_cache(ruta, anos_interes); metodo = 'pivot_cache_xml'
    except (BadZipFile, KeyError, ValueError, OSError, ImportError):
        partes = []
        for hoja, bloque in pd.read_excel(ruta, sheet_name=None).items():
            try: campos = resolver_campos(list(bloque.columns))
            except ValueError: continue
            bloque = bloque.rename(columns={v: k for k, v in campos.items()}); bloque['tipo'] = bloque['tipo'].map(normalizar_texto); bloque['ano'] = bloque['ano'].map(convertir_entero)
            partes.append(bloque[(bloque['tipo'] == 'ESTIMADO') & bloque['ano'].isin(anos_interes)])
        if not partes: raise ValueError('No se encontró una hoja legible')
        tabla = pd.concat(partes, ignore_index=True); metodo = 'hojas_excel_fallback'
    tabla['ano'] = tabla['ano'].map(convertir_entero); tabla['semana'] = tabla['semana'].map(convertir_entero); return tabla, metodo

## 3. Extracción H1-H5, trazabilidad y salidas

Cada solicitud construye cinco semanas objetivo desde la semana solicitada. El archivo usado puede ser posterior, pero no se cambia la semana solicitada.

In [ ]:
def construir_horizontes(ano, semana):
    lunes = date.fromisocalendar(int(ano), int(semana), 1)
    return pd.DataFrame([{'anio_objetivo': (lunes + timedelta(weeks=i)).isocalendar().year, 'semana_objetivo': (lunes + timedelta(weeks=i)).isocalendar().week, 'horizonte': i + 1, 'semanas_adelante': i, 'lunes_objetivo': pd.Timestamp(lunes + timedelta(weeks=i))} for i in range(NUMERO_HORIZONTES)])

def preparar_h1_h5(base, fila):
    ano = int(fila['ano_inicio']); semana = int(fila['semana_solicitada']); h = construir_horizontes(ano, semana); tabla = base.copy()
    tabla['ano'] = tabla['ano'].map(convertir_entero); tabla['semana'] = tabla['semana'].map(convertir_entero)
    resultado = tabla.merge(h, left_on=['ano', 'semana'], right_on=['anio_objetivo', 'semana_objetivo'], how='inner')
    resultado['anio_inicial_estimado'] = ano; resultado['semana_inicial_estimado'] = semana; resultado['lunes_inicial_estimado'] = pd.Timestamp(date.fromisocalendar(ano, semana, 1))
    resultado['archivo_origen'] = fila['archivo']; resultado['ruta_relativa_origen'] = fila['ruta_relativa']; resultado['carpeta_origen'] = fila['carpeta']; resultado['semana_archivo'] = int(fila['semana_inicio'])
    resultado['archivo_es_posterior'] = bool(fila['archivo_es_posterior']); resultado['diferencia_semanas_archivo'] = int(fila['diferencia_semanas_archivo']); resultado['metodo_lectura'] = fila.get('metodo_lectura', '')
    return resultado

def procesar_solicitud(fila):
    h = construir_horizontes(fila['ano_inicio'], fila['semana_solicitada']); anos = set(h['anio_objetivo'].astype(int))
    try:
        base, metodo = leer_estimados_archivo(Path(fila['ruta_absoluta']), anos); fila = fila.copy(); fila['metodo_lectura'] = metodo; salida = preparar_h1_h5(base, fila)
        control = {**fila[['ano_inicio', 'semana_solicitada', 'semana_inicio', 'archivo', 'ruta_relativa']].to_dict(), 'metodo_lectura': metodo, 'filas_cache': len(base), 'filas_h1_h5': len(salida), 'estado': 'ok' if len(salida) else 'sin filas H1-H5'}
        return salida, control, None
    except Exception as e:
        return pd.DataFrame(), {'ano_inicio': fila['ano_inicio'], 'semana_solicitada': fila['semana_solicitada'], 'semana_inicio': fila['semana_inicio'], 'archivo': fila['archivo'], 'ruta_relativa': fila['ruta_relativa'], 'estado': 'error'}, {'archivo': fila['archivo'], 'detalle_error': f'{type(e).__name__}: {e}'}

partes, controles, errores = [], [], []
with ThreadPoolExecutor(max_workers=4) as executor:
    futuros = [executor.submit(procesar_solicitud, fila) for _, fila in plan_extraccion.iterrows()]
    for i, futuro in enumerate(as_completed(futuros), 1):
        parte, control, error = futuro.result(); partes.append(parte); controles.append(control)
        if error: errores.append(error)
        if i % 10 == 0 or i == len(futuros): print(f'Procesadas {i}/{len(futuros)} solicitudes')
estimados_crudos_consolidados = pd.concat(partes, ignore_index=True) if partes else pd.DataFrame()
control_extraccion = pd.DataFrame(controles); errores_lectura = pd.DataFrame(errores); CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
tablas_salida = {'inventario_archivos': inventario_archivos, 'archivos_seleccionados': archivos_seleccionados, 'archivos_duplicados': archivos_duplicados, 'archivos_excluidos': archivos_excluidos, 'plan_extraccion': plan_extraccion, 'control_extraccion': control_extraccion, 'errores_lectura': errores_lectura}
for nombre, tabla in tablas_salida.items(): tabla.to_csv(CARPETA_SALIDA / f'{nombre}.csv', index=False, encoding='utf-8-sig')
estimados_crudos_consolidados.to_csv(CARPETA_SALIDA / 'estimados_crudos_h1_h5.csv', index=False, encoding='utf-8-sig')
print(f'Filas H1-H5 extraídas: {len(estimados_crudos_consolidados):,}'); print('Errores de lectura:', len(errores_lectura)); display(estimados_crudos_consolidados.head())

## 4. EDA de faltantes y curvas por año/semana

Estas tablas muestran semanas recuperadas con el siguiente archivo, errores de lectura y horizontes que no aparecieron.

In [ ]:
plan_control = plan_extraccion[['ano_inicio', 'semana_solicitada', 'semana_inicio', 'archivo_es_posterior', 'diferencia_semanas_archivo']].drop_duplicates()
display(plan_control[plan_control['archivo_es_posterior']].sort_values(['ano_inicio', 'semana_solicitada']).head(30))
observado = estimados_crudos_consolidados[['anio_inicial_estimado', 'semana_inicial_estimado', 'horizonte']].drop_duplicates()
esperado = plan_control.rename(columns={'ano_inicio': 'anio_inicial_estimado', 'semana_solicitada': 'semana_inicial_estimado'})[['anio_inicial_estimado', 'semana_inicial_estimado']].assign(_k=1).merge(pd.DataFrame({'horizonte': range(1, 6), '_k': 1}), on='_k').drop(columns='_k')
faltantes_horizontes = esperado.merge(observado, how='left', indicator=True); faltantes_horizontes = faltantes_horizontes[faltantes_horizontes['_merge'].eq('left_only')].drop(columns='_merge')
display(faltantes_horizontes.head(30))
resumen_semanal = estimados_crudos_consolidados.groupby(['anio_inicial_estimado', 'semana_inicial_estimado'], as_index=False).agg(filas=('horizonte', 'size'), horizontes=('horizonte', 'nunique'), semanas_objetivo=('semana_objetivo', 'nunique'))
display(resumen_semanal.tail(20))
import matplotlib.pyplot as plt
columna_tallos = 'tallos' if 'tallos' in estimados_crudos_consolidados.columns else 'ESTI. EXPORTABLE'
curva = estimados_crudos_consolidados.groupby(['anio_objetivo', 'semana_objetivo'], as_index=False)[columna_tallos].sum(); curva['periodo'] = curva['anio_objetivo'].astype(str) + '-S' + curva['semana_objetivo'].astype(str).str.zfill(2)
plt.figure(figsize=(14, 5)); plt.plot(curva['periodo'], curva[columna_tallos], marker='o', linewidth=1); plt.xticks(rotation=90); plt.title('Curva de estimados por año/semana objetivo'); plt.xlabel('Año-semana ISO'); plt.ylabel('Tallos estimados'); plt.tight_layout(); plt.show()